In [1]:
# Complete code with all necessary imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../db/SGJobData_cleaned.csv.gz");
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1044597 entries, 0 to 1044596
Data columns (total 32 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   employmentTypes                     1044597 non-null  object 
 1   metadata_expiryDate                 1044597 non-null  object 
 2   metadata_isPostedOnBehalf           1044597 non-null  bool   
 3   metadata_jobPostId                  1044597 non-null  object 
 4   metadata_newPostingDate             1044597 non-null  object 
 5   metadata_originalPostingDate        1044597 non-null  object 
 6   metadata_repostCount                1044597 non-null  int64  
 7   metadata_totalNumberJobApplication  1044597 non-null  int64  
 8   metadata_totalNumberOfView          1044597 non-null  int64  
 9   minimumYearsExperience              1044597 non-null  int64  
 10  numberOfVacancies                   1044597 non-null  int64  
 11  positionLev

In [13]:
def q1_salary_benchmarking_numbers(df, search_keyword=None, return_data=False):
    """
    Q1: What salary should we offer for role X?
    Returns comprehensive salary benchmarks by position level and industry
    
    Parameters:
    -----------
    df : DataFrame
        The salary dataset
    search_keyword : str, optional
        Specific role to search for (e.g., 'data scientist')
    return_data : bool, default=False
        If True, returns the dataframes instead of printing
    
    Returns:
    --------
    tuple : (salary_by_position, salary_by_industry) if return_data=True
    """
    
    # Helper function for consistent printing
    def print_header(title, char="="):
        print(char * 80)
        print(title)
        print(char * 80)
    
    def print_subheader(title):
        print(f"\n📊 {title}")
        print("-" * 80)
    
    # Print main header
    print_header("Q1: SALARY BENCHMARKING - WHAT SHOULD WE OFFER FOR ROLE X?")
    
    # 1. Salary benchmarks by position level
    print_subheader("SALARY BENCHMARKS BY POSITION LEVEL")
    
    # Define quantile function for cleaner code
    def quantile_10(x): return x.quantile(0.10)
    def quantile_25(x): return x.quantile(0.25)
    def quantile_50(x): return x.quantile(0.50)
    def quantile_75(x): return x.quantile(0.75)
    def quantile_90(x): return x.quantile(0.90)
    
    salary_by_position = (df.groupby('positionLevels')['average_salary']
                          .agg([
                              ('Count', 'count'),
                              ('Min', 'min'),
                              ('P10', quantile_10),
                              ('P25', quantile_25),
                              ('Median_P50', quantile_50),
                              ('P75', quantile_75),
                              ('P90', quantile_90),
                              ('Max', 'max'),
                              ('Mean', 'mean'),
                              ('Std', 'std')
                          ])
                          .round(0)
                          .sort_values('Median_P50', ascending=False))
    
    print(salary_by_position)
    
    # 2. Salary benchmarks by industry (top 10)
    print_subheader("TOP 10 INDUSTRIES BY AVERAGE SALARY")
    
    salary_by_industry = (df.groupby('industry_primary')['average_salary']
                          .agg([
                              ('Count', 'count'),
                              ('Median_P50', quantile_50),
                              ('P25', quantile_25),
                              ('P75', quantile_75),
                              ('Mean', 'mean')
                          ])
                          .round(0)
                          .sort_values('Median_P50', ascending=False)
                          .head(10))
    
    print(salary_by_industry)
    
    # 3. Specific role search (if keyword provided)
    if search_keyword:
        print_header(f"🔍 SEARCH RESULTS FOR: '{search_keyword}'", char="=")
        search_results = search_role(df, search_keyword)
    else:
        print_header("💡 TIP: Use search_keyword parameter to search for specific roles", char="-")
        print("Example: q1_salary_benchmarking_numbers(df, search_keyword='data scientist')")
    
    # Return data if requested
    if return_data:
        return salary_by_position, salary_by_industry


def search_role(df, keyword):
    """
    Search for specific roles by keyword (case-insensitive)
    
    Parameters:
    -----------
    df : DataFrame
        The salary dataset
    keyword : str
        Role to search for
    
    Returns:
    --------
    DataFrame : Matching rows or None if no matches found
    """
    # Case-insensitive search
    matching = df[df['title'].str.contains(keyword, case=False, na=False)]
    
    if len(matching) == 0:
        print(f"❌ No jobs found matching '{keyword}'")
        return None
    
    # Print results
    print(f"\n✅ Found {len(matching):,} jobs matching '{keyword}'")
    print("\n" + "-" * 40)
    
    # Salary summary
    print("\n💰 SALARY SUMMARY:")
    print(matching['average_salary'].describe().round(0))
    
    # Position level distribution
    print("\n📊 POSITION LEVEL DISTRIBUTION:")
    print(matching['positionLevels'].value_counts().head())
    
    # Top industries
    print("\n🏢 TOP INDUSTRIES:")
    print(matching['industry_primary'].value_counts().head())
    
    # Salary by position level
    print("\n📈 SALARY BY POSITION LEVEL:")
    salary_by_level = (matching.groupby('positionLevels')['average_salary']
                       .agg([
                           ('Count', 'count'),
                           ('Median', 'median'),
                           ('P25', lambda x: x.quantile(0.25)),
                           ('P75', lambda x: x.quantile(0.75))
                       ])
                       .round(0))
    print(salary_by_level)
    
    return matching

q1_salary_benchmarking_numbers(df)

Q1: SALARY BENCHMARKING - WHAT SHOULD WE OFFER FOR ROLE X?

📊 SALARY BENCHMARKS BY POSITION LEVEL
--------------------------------------------------------------------------------
                    Count  Min     P10     P25  Median_P50      P75      P90  \
positionLevels                                                                 
Senior Management   22807  1.0  5000.0  7750.0      9500.0  13500.0  18000.0   
Middle Management   27375  1.0  3650.0  4000.0      6500.0   9500.0  14000.0   
Manager            110122  1.0  3800.0  4650.0      6000.0   8000.0  10500.0   
Professional       112208  1.0  3500.0  4050.0      6000.0   8750.0  12000.0   
Senior Executive   100459  1.0  3500.0  4000.0      5000.0   6500.0   8750.0   
Executive          253701  1.0  2750.0  3150.0      3750.0   4500.0   6000.0   
Junior Executive   167656  1.0  2250.0  2650.0      3150.0   3750.0   4700.0   
Non-executive      131608  1.0  1850.0  2200.0      2700.0   3500.0   4250.0   
Fresh/entry level  11

In [3]:
def q2_hard_to_fill_numbers(df):
    """
    Q2: Which roles are hard to fill?
    Analyzes hard to fill patterns by: Industry, Position Level, Employment Type, Experience, Salary
    Does NOT use titles (avoid regex complexity)
    """
    
    print("="*80)
    print("Q2: HARD TO FILL ROLES ANALYSIS - BY CATEGORICAL DIMENSIONS")
    print("="*80)
    
    # Filter to only jobs with activity data
    df_valid = df[
        (df['metadata_totalNumberJobApplication'] > 0) | 
        (df['metadata_repostCount'] > 0) |
        (df['applications_per_vacancy'] > 0)
    ].copy()
    
    print(f"\n📊 DATA FILTERING")
    print("-"*80)
    print(f"Total Jobs: {len(df):,}")
    print(f"Jobs with activity data: {len(df_valid):,} ({len(df_valid)/len(df)*100:.1f}%)")
    print(f"Jobs EXCLUDED (no activity): {len(df) - len(df_valid):,} (new/untracked postings)")
    
    if len(df_valid) == 0:
        print("\n⚠️ No jobs with activity data found. Cannot analyze hard to fill roles.")
        return None
    
    # Calculate median values from valid data
    median_reposts = df_valid['metadata_repostCount'].median()
    median_apps_per_vac = df_valid['applications_per_vacancy'].median()
    
    print(f"\n📊 KEY METRICS (from valid data)")
    print("-"*80)
    print(f"Median Repost Count: {median_reposts:.0f}")
    print(f"Median Applications per Vacancy: {median_apps_per_vac:.2f}")
    
    # Classify hard to fill
    df_valid['is_hard_to_fill'] = (
        (df_valid['metadata_repostCount'] >= median_reposts) & 
        (df_valid['applications_per_vacancy'] <= median_apps_per_vac)
    )
    
    htf_roles = df_valid[df_valid['is_hard_to_fill']]
    not_htf = df_valid[~df_valid['is_hard_to_fill']]
    
    print(f"\n📊 HARD TO FILL SUMMARY")
    print("-"*80)
    print(f"Hard to Fill Jobs: {len(htf_roles):,} ({len(htf_roles)/len(df_valid)*100:.1f}% of active jobs)")
    print(f"Average Reposts (Hard to Fill): {htf_roles['metadata_repostCount'].mean():.1f}")
    print(f"Average Apps/Vacancy (Hard to Fill): {htf_roles['applications_per_vacancy'].mean():.2f}")
    
    # ============================================================================
    # ANALYSIS BY EACH DIMENSION
    # ============================================================================
    
    # 1. ANALYSIS BY INDUSTRY
    print("\n" + "="*80)
    print("📊 1. HARD TO FILL BY INDUSTRY")
    print("="*80)
    
    industry_htf = df_valid.groupby('industry_primary').agg(
        Total_Jobs=('is_hard_to_fill', 'count'),
        Hard_to_Fill_Count=('is_hard_to_fill', 'sum'),
        Pct_Hard_to_Fill=('is_hard_to_fill', 'mean'),
        Avg_Reposts=('metadata_repostCount', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean'),
        Avg_Salary=('average_salary', 'mean'),
        Avg_Experience=('minimumYearsExperience', 'mean')
    ).round(2)
    
    industry_htf['Pct_Hard_to_Fill'] = industry_htf['Pct_Hard_to_Fill'] * 100
    
    # Filter to industries with at least 50 jobs for statistical significance
    industry_htf = industry_htf[industry_htf['Total_Jobs'] >= 50]
    industry_htf = industry_htf.sort_values('Pct_Hard_to_Fill', ascending=False)
    
    print("\nTop 10 Industries with Highest % Hard to Fill:")
    print(industry_htf.head(10))
    
    print("\nBottom 10 Industries with Lowest % Hard to Fill:")
    print(industry_htf.tail(10))
    
    # ============================================================================
    # 2. ANALYSIS BY POSITION LEVEL
    print("\n" + "="*80)
    print("📊 2. HARD TO FILL BY POSITION LEVEL")
    print("="*80)
    
    position_htf = df_valid.groupby('positionLevels').agg(
        Total_Jobs=('is_hard_to_fill', 'count'),
        Hard_to_Fill_Count=('is_hard_to_fill', 'sum'),
        Pct_Hard_to_Fill=('is_hard_to_fill', 'mean'),
        Avg_Reposts=('metadata_repostCount', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean'),
        Avg_Salary=('average_salary', 'mean'),
        Avg_Experience=('minimumYearsExperience', 'mean')
    ).round(2)
    
    position_htf['Pct_Hard_to_Fill'] = position_htf['Pct_Hard_to_Fill'] * 100
    position_htf = position_htf.sort_values('Pct_Hard_to_Fill', ascending=False)
    
    print(position_htf)
    
    # ============================================================================
    # 3. ANALYSIS BY EMPLOYMENT TYPE
    print("\n" + "="*80)
    print("📊 3. HARD TO FILL BY EMPLOYMENT TYPE")
    print("="*80)
    
    employment_htf = df_valid.groupby('employmentTypes').agg(
        Total_Jobs=('is_hard_to_fill', 'count'),
        Hard_to_Fill_Count=('is_hard_to_fill', 'sum'),
        Pct_Hard_to_Fill=('is_hard_to_fill', 'mean'),
        Avg_Reposts=('metadata_repostCount', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean'),
        Avg_Salary=('average_salary', 'mean')
    ).round(2)
    
    employment_htf['Pct_Hard_to_Fill'] = employment_htf['Pct_Hard_to_Fill'] * 100
    employment_htf = employment_htf.sort_values('Pct_Hard_to_Fill', ascending=False)
    
    print(employment_htf)
    
    # ============================================================================
    # 4. ANALYSIS BY MINIMUM YEARS EXPERIENCE
    print("\n" + "="*80)
    print("📊 4. HARD TO FILL BY MINIMUM YEARS EXPERIENCE")
    print("="*80)
    
    # Group experience into buckets
    df_valid['exp_bucket'] = pd.cut(
        df_valid['minimumYearsExperience'],
        bins=[-1, 0, 2, 5, 10, 20, 100],
        labels=['0 years', '1-2 years', '3-5 years', '6-10 years', '11-20 years', '20+ years']
    )
    
    experience_htf = df_valid.groupby('exp_bucket').agg(
        Total_Jobs=('is_hard_to_fill', 'count'),
        Hard_to_Fill_Count=('is_hard_to_fill', 'sum'),
        Pct_Hard_to_Fill=('is_hard_to_fill', 'mean'),
        Avg_Reposts=('metadata_repostCount', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean'),
        Avg_Salary=('average_salary', 'mean')
    ).round(2)
    
    experience_htf['Pct_Hard_to_Fill'] = experience_htf['Pct_Hard_to_Fill'] * 100
    experience_htf = experience_htf.sort_values('Pct_Hard_to_Fill', ascending=False)
    
    print(experience_htf)
    
    # ============================================================================
    # 5. ANALYSIS BY SALARY BAND
    print("\n" + "="*80)
    print("📊 5. HARD TO FILL BY SALARY BAND")
    print("="*80)
    
    salary_band_htf = df_valid.groupby('salary_band').agg(
        Total_Jobs=('is_hard_to_fill', 'count'),
        Hard_to_Fill_Count=('is_hard_to_fill', 'sum'),
        Pct_Hard_to_Fill=('is_hard_to_fill', 'mean'),
        Avg_Reposts=('metadata_repostCount', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean'),
        Avg_Salary=('average_salary', 'mean')
    ).round(2)
    
    salary_band_htf['Pct_Hard_to_Fill'] = salary_band_htf['Pct_Hard_to_Fill'] * 100
    salary_band_htf = salary_band_htf.sort_values('Pct_Hard_to_Fill', ascending=False)
    
    print(salary_band_htf)
    
    # ============================================================================
    # 6. COMBINED ANALYSIS - TOP RISK FACTORS
    print("\n" + "="*80)
    print("📊 6. TOP HARD TO FILL COMBINATIONS (Industry + Position Level)")
    print("="*80)
    
    # Find the most challenging combinations
    combo_htf = df_valid.groupby(['industry_primary', 'positionLevels']).agg(
        Total_Jobs=('is_hard_to_fill', 'count'),
        Pct_Hard_to_Fill=('is_hard_to_fill', 'mean'),
        Avg_Reposts=('metadata_repostCount', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean'),
        Avg_Salary=('average_salary', 'mean')
    ).round(2)
    
    combo_htf['Pct_Hard_to_Fill'] = combo_htf['Pct_Hard_to_Fill'] * 100
    combo_htf = combo_htf[combo_htf['Total_Jobs'] >= 30]  # Minimum sample size
    combo_htf = combo_htf.sort_values('Pct_Hard_to_Fill', ascending=False).head(20)
    
    print(combo_htf)
    
    # ============================================================================
    # 7. KEY INSIGHTS & RECOMMENDATIONS
    print("\n" + "="*80)
    print("💡 KEY INSIGHTS: WHY ARE THESE ROLES HARD TO FILL?")
    print("="*80)
    
    # Compare hard to fill vs others
    print("\n📊 COMPARISON: HARD TO FILL VS OTHER ROLES")
    print("-"*80)
    
    comparisons = {
        'Average Salary': ('average_salary', '${:,.0f}'),
        'Avg Experience Required': ('minimumYearsExperience', '{:.1f} years'),
        'Avg Vacancies per Posting': ('numberOfVacancies', '{:.1f}'),
        'Avg Applications': ('metadata_totalNumberJobApplication', '{:.1f}')
    }
    
    for metric, (col, fmt) in comparisons.items():
        htf_val = htf_roles[col].mean()
        not_htf_val = not_htf[col].mean()
        diff = htf_val - not_htf_val
        diff_pct = ((diff / not_htf_val) * 100) if not_htf_val != 0 else 0
        
        print(f"\n{metric}:")
        print(f"  Hard to Fill: {fmt.format(htf_val)}")
        print(f"  Other Roles: {fmt.format(not_htf_val)}")
        print(f"  Difference: {diff_pct:+.1f}% {'higher' if diff > 0 else 'lower'}")
    
    # Top risk factors summary
    print("\n" + "="*80)
    print("🎯 TOP RISK FACTORS FOR HARD TO FILL ROLES")
    print("="*80)
    
    # Find industries with highest hard to fill rates
    top_industries = industry_htf.head(3)['Pct_Hard_to_Fill']
    print(f"\n🏭 Industries most likely to be hard to fill:")
    for industry, pct in top_industries.items():
        print(f"  • {industry}: {pct:.1f}% hard to fill")
    
    # Find position levels with highest hard to fill rates
    top_positions = position_htf.head(3)['Pct_Hard_to_Fill']
    print(f"\n👔 Position levels most likely to be hard to fill:")
    for level, pct in top_positions.items():
        print(f"  • {level}: {pct:.1f}% hard to fill")
    
    # Find salary bands with highest hard to fill rates
    top_salary_bands = salary_band_htf.head(3)['Pct_Hard_to_Fill']
    print(f"\n💰 Salary bands most likely to be hard to fill:")
    for band, pct in top_salary_bands.items():
        print(f"  • {band}: {pct:.1f}% hard to fill")
    
    # Find experience buckets with highest hard to fill rates
    top_experience = experience_htf.head(3)['Pct_Hard_to_Fill']
    print(f"\n📅 Experience requirements most likely to be hard to fill:")
    for exp, pct in top_experience.items():
        print(f"  • {exp}: {pct:.1f}% hard to fill")
    
    # Employment type risks
    top_employment = employment_htf.head(3)['Pct_Hard_to_Fill']
    print(f"\n📋 Employment types most likely to be hard to fill:")
    for emp_type, pct in top_employment.items():
        print(f"  • {emp_type}: {pct:.1f}% hard to fill")
    
    # Recommendations
    print("\n" + "="*80)
    print("💡 RECOMMENDATIONS FOR HARD TO FILL ROLES")
    print("="*80)
    
    # Find the most risky combination
    if len(combo_htf) > 0:
        riskiest = combo_htf.iloc[0]
        print(f"\n🚨 HIGHEST RISK COMBINATION:")
        print(f"  • Industry: {combo_htf.index[0][0]}")
        print(f"  • Position Level: {combo_htf.index[0][1]}")
        print(f"  • Hard to Fill Rate: {riskiest['Pct_Hard_to_Fill']:.1f}%")
        print(f"  • Average Salary: ${riskiest['Avg_Salary']:,.0f}")
    
    print("\n📌 Action Items:")
    print("  1. Review compensation for roles in high-risk industries and position levels")
    print("  2. Consider adjusting experience requirements if they exceed market norms")
    print("  3. Focus recruitment efforts on industries with lower hard-to-fill rates")
    print("  4. Evaluate if employment type (e.g., contract, temporary) is affecting attractiveness")
    print("  5. Benchmark salary bands against market rates")
    
    return htf_roles

# Run Q2
q2_hard_to_fill_numbers(df)

Q2: HARD TO FILL ROLES ANALYSIS - BY CATEGORICAL DIMENSIONS

📊 DATA FILTERING
--------------------------------------------------------------------------------
Total Jobs: 1,044,597
Jobs with activity data: 392,929 (37.6%)
Jobs EXCLUDED (no activity): 651,668 (new/untracked postings)

📊 KEY METRICS (from valid data)
--------------------------------------------------------------------------------
Median Repost Count: 0
Median Applications per Vacancy: 1.00

📊 HARD TO FILL SUMMARY
--------------------------------------------------------------------------------
Hard to Fill Jobs: 269,166 (68.5% of active jobs)
Average Reposts (Hard to Fill): 0.1
Average Apps/Vacancy (Hard to Fill): 0.75

📊 1. HARD TO FILL BY INDUSTRY

Top 10 Industries with Highest % Hard to Fill:
                            Total_Jobs  Hard_to_Fill_Count  Pct_Hard_to_Fill  \
industry_primary                                                               
Personal Care / Beauty            1211                1031           

,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,minimumYearsExperience,...,is_selective_hire,industry_list,industry_primary,posting_year,posting_month,posting_quarter,posting_weekday,days_to_expiry,is_agency_post,posting_month_year
4,Full Time,2023-05-08,False,MCF-2023-0273976,2023-04-08,2023-04-08,0,3,99,2,...,False,Admin / Secretarial,Admin / Secretarial,2023,4,2,5,30,False,2023-04
15,Full Time,2023-05-08,False,MCF-2023-0273986,2023-04-08,2023-04-08,0,3,142,15,...,False,Telecommunications,Telecommunications,2023,4,2,5,30,False,2023-04
18,Permanent,2023-05-21,False,MCF-2023-0261992,2023-04-21,2023-04-04,2,0,105,1,...,False,"Engineering,Repair and Maintenance,Precision E...",Engineering,2023,4,2,1,47,False,2023-04
28,Full Time,2023-05-08,True,MCF-2023-0274189,2023-04-08,2023-04-08,0,3,97,10,...,False,General Work,General Work,2023,4,2,5,30,True,2023-04
37,Contract,2023-05-12,False,MCF-2023-0261995,2023-04-12,2023-04-04,2,0,66,0,...,False,"Admin / Secretarial,Building and Construction,...",Admin / Secretarial,2023,4,2,1,38,False,2023-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044588,Internship/Attachment,2023-10-03,False,RANDOM_JOB_20251115011346673349_1,2023-03-09,2023-03-02,0,156,2559,63,...,False,"Customer Service,Human Resources,Professional ...",Customer Service,2023,3,1,3,215,False,2023-03
1044589,Part Time,2024-06-18,True,RANDOM_JOB_20251115011347120191_2,2023-12-05,2023-12-05,0,626,192,60,...,False,"Admin / Secretarial,Architecture / Interior De...",Admin / Secretarial,2023,12,4,1,196,True,2023-12
1044593,Freelance,2024-03-18,False,RANDOM_JOB_20251115011348553903_6,2023-09-04,2023-08-12,1,131,1626,8,...,False,"General Management,Professional Services,Sales...",General Management,2023,8,3,5,219,False,2023-08
1044595,Internship/Attachment,2024-10-11,False,RANDOM_JOB_20251115011349285489_8,2024-01-22,2024-01-03,0,465,2281,59,...,False,"Admin / Secretarial,Building and Construction,...",Admin / Secretarial,2024,1,1,2,282,False,2024-01


In [4]:
def q3_high_demand_numbers(df):
    """
    Q3: Which roles/industries have the most demand?
    Identifies top industries and roles by total vacancies
    """
    
    print("="*80)
    print("Q3: HIGH DEMAND ROLES AND INDUSTRIES")
    print("="*80)
    
    # 1. Top industries by vacancies - FIXED
    print("\n📊 TOP 10 INDUSTRIES BY TOTAL VACANCIES")
    print("-"*80)
    
    # Create aggregation properly with reset_index
    industry_demand = df.groupby('industry_primary').agg(
        Total_Vacancies=('numberOfVacancies', 'sum'),
        Avg_Vacancies_Per_Posting=('numberOfVacancies', 'mean'),
        Job_Postings=('title', 'count'),
        Avg_Salary=('average_salary', 'mean')
    ).round(2)
    
    industry_demand = industry_demand.sort_values('Total_Vacancies', ascending=False).head(10)
    
    # Add percentage
    total_vacancies = industry_demand['Total_Vacancies'].sum()
    industry_demand['Pct_of_Total_Vacancies'] = (industry_demand['Total_Vacancies'] / total_vacancies * 100).round(1)
    
    print(industry_demand)
    
    # 2. Top roles by vacancies - FIXED
    print("\n📊 TOP 10 ROLES BY TOTAL VACANCIES")
    print("-"*80)
    
    role_demand = df.groupby('title').agg(
        Total_Vacancies=('numberOfVacancies', 'sum'),
        Avg_Vacancies_Per_Posting=('numberOfVacancies', 'mean'),
        Job_Postings=('title', 'count'),
        Avg_Salary=('average_salary', 'mean'),
        Typical_Level=('positionLevels', lambda x: x.mode()[0] if len(x) > 0 else 'N/A')
    ).round(2)
    
    role_demand = role_demand.sort_values('Total_Vacancies', ascending=False).head(10)
    
    print(role_demand)
    
    # 3. Vacancies by position level - FIXED
    print("\n📊 VACANCIES BY POSITION LEVEL")
    print("-"*80)
    
    position_demand = df.groupby('positionLevels').agg(
        Total_Vacancies=('numberOfVacancies', 'sum'),
        Avg_Vacancies_Per_Posting=('numberOfVacancies', 'mean'),
        Job_Postings=('title', 'count'),
        Avg_Salary=('average_salary', 'mean'),
        Avg_Apps_Per_Vac=('applications_per_vacancy', 'mean')
    ).round(2)
    
    position_demand = position_demand.sort_values('Total_Vacancies', ascending=False)
    
    print(position_demand)
    
    # 4. Industry + Position Level combinations - FIXED
    print("\n📊 TOP 10 INDUSTRY-POSITION COMBINATIONS BY VACANCIES")
    print("-"*80)
    
    combo_demand = df.groupby(['industry_primary', 'positionLevels']).agg(
        Total_Vacancies=('numberOfVacancies', 'sum'),
        Job_Postings=('title', 'count'),
        Avg_Salary=('average_salary', 'mean')
    ).round(2)
    
    combo_demand = combo_demand.sort_values('Total_Vacancies', ascending=False).head(10)
    
    print(combo_demand)
    
    # 5. Growth indicators - FIXED
    print("\n📊 ROLES WITH HIGHEST VACANCIES PER POSTING (Multiple positions per ad)")
    print("-"*80)
    
    high_volume_roles = df.groupby('title').agg(
        Total_Vacancies=('numberOfVacancies', 'sum'),
        Avg_Vacancies_Per_Posting=('numberOfVacancies', 'mean'),
        Postings=('title', 'count')
    ).round(2)
    
    high_volume_roles = high_volume_roles[high_volume_roles['Postings'] >= 5]
    high_volume_roles = high_volume_roles.sort_values('Avg_Vacancies_Per_Posting', ascending=False).head(10)
    
    print(high_volume_roles)
    
    # Key insights
    print("\n" + "="*80)
    print("💡 KEY DEMAND INSIGHTS")
    print("="*80)
    
    print(f"\n🏢 Top Industry: {industry_demand.index[0]} ({industry_demand.iloc[0]['Total_Vacancies']:.0f} vacancies)")
    print(f"   Accounting for {industry_demand.iloc[0]['Pct_of_Total_Vacancies']:.1f}% of all vacancies")
    
    print(f"\n💼 Top Role: {role_demand.index[0]} ({role_demand.iloc[0]['Total_Vacancies']:.0f} vacancies)")
    print(f"   Typical Level: {role_demand.iloc[0]['Typical_Level']}")
    
    print(f"\n👔 Top Position Level: {position_demand.index[0]} ({position_demand.iloc[0]['Total_Vacancies']:.0f} vacancies)")
    
    return industry_demand, role_demand

# Run Q3
q3_high_demand_numbers(df)

Q3: HIGH DEMAND ROLES AND INDUSTRIES

📊 TOP 10 INDUSTRIES BY TOTAL VACANCIES
--------------------------------------------------------------------------------
                                  Total_Vacancies  Avg_Vacancies_Per_Posting  \
industry_primary                                                               
F&B                                        244260                       4.09   
Customer Service                           240390                       3.71   
Information Technology                     236667                       2.36   
Admin / Secretarial                        229399                       2.23   
Engineering                                190675                       1.91   
Healthcare / Pharmaceutical                168524                       5.07   
Accounting / Auditing / Taxation           164882                       2.10   
Building and Construction                  159855                       2.16   
Education and Training                    

(                                  Total_Vacancies  Avg_Vacancies_Per_Posting  \
 industry_primary                                                               
 F&B                                        244260                       4.09   
 Customer Service                           240390                       3.71   
 Information Technology                     236667                       2.36   
 Admin / Secretarial                        229399                       2.23   
 Engineering                                190675                       1.91   
 Healthcare / Pharmaceutical                168524                       5.07   
 Accounting / Auditing / Taxation           164882                       2.10   
 Building and Construction                  159855                       2.16   
 Education and Training                     156371                       4.39   
 Banking and Finance                        110853                       2.38   
 
                          

In [6]:
def q5_posting_timing_numbers(df):
    """
    Q5: When should we post jobs?
    Analyzes posting patterns and application volumes over time
    """
    
    print("="*80)
    print("Q5: OPTIMAL POSTING TIMING ANALYSIS")
    print("="*80)
    
    # Convert to datetime
    df['posting_date'] = pd.to_datetime(df['metadata_originalPostingDate'])
    
    # Filter from May 2023
    df_filtered = df[df['posting_date'] >= '2023-05-01'].copy()
    
    print(f"\n📊 TIME PERIOD OVERVIEW")
    print("-"*80)
    print(f"Data from: {df_filtered['posting_date'].min().date()} to {df_filtered['posting_date'].max().date()}")
    print(f"Total Postings in Period: {len(df_filtered):,}")
    
    # Monthly summary
    df_filtered['posting_month'] = df_filtered['posting_date'].dt.to_period('M')
    monthly = df_filtered.groupby('posting_month').agg({
        'title': 'count',
        'metadata_totalNumberJobApplication': 'sum',
        'average_salary': 'mean',
        'numberOfVacancies': 'sum',
        'applications_per_vacancy': 'mean'
    }).rename(columns={'title': 'posting_count'})
    
    monthly['apps_per_posting'] = monthly['metadata_totalNumberJobApplication'] / monthly['posting_count']
    monthly.reset_index(inplace=True)
    monthly['month_str'] = monthly['posting_month'].astype(str)
    
    print("\n📊 MONTHLY TRENDS")
    print("-"*80)
    print(monthly[['month_str', 'posting_count', 'metadata_totalNumberJobApplication', 
                   'apps_per_posting', 'average_salary']].round(2))
    
    # Best and worst months
    print("\n📊 BEST AND WORST MONTHS")
    print("-"*80)
    
    best_posting = monthly.loc[monthly['posting_count'].idxmax()]
    worst_posting = monthly.loc[monthly['posting_count'].idxmin()]
    best_apps = monthly.loc[monthly['metadata_totalNumberJobApplication'].idxmax()]
    best_engagement = monthly.loc[monthly['apps_per_posting'].idxmax()]
    
    print(f"🔝 Best Month for Postings: {best_posting['month_str']} ({best_posting['posting_count']:,} postings)")
    print(f"📉 Worst Month for Postings: {worst_posting['month_str']} ({worst_posting['posting_count']:,} postings)")
    print(f"📊 Best Month for Applications: {best_apps['month_str']} ({best_apps['metadata_totalNumberJobApplication']:,} applications)")
    print(f"⭐ Best Engagement Month: {best_engagement['month_str']} ({best_engagement['apps_per_posting']:.1f} apps/posting)")
    
    # Weekly patterns
    print("\n📊 WEEKLY POSTING PATTERNS")
    print("-"*80)
    
    df_filtered['weekday'] = df_filtered['posting_date'].dt.day_name()
    weekday_stats = df_filtered.groupby('weekday').agg({
        'title': 'count',
        'metadata_totalNumberJobApplication': 'sum',
        'applications_per_vacancy': 'mean'
    }).rename(columns={'title': 'postings'})
    
    weekday_stats['apps_per_posting'] = weekday_stats['metadata_totalNumberJobApplication'] / weekday_stats['postings']
    
    # Order weekdays
    week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekday_stats = weekday_stats.reindex(week_order)
    
    print(weekday_stats.round(2))
    
    # Best day to post
    best_day_posting = weekday_stats['postings'].idxmax()
    best_day_engagement = weekday_stats['apps_per_posting'].idxmax()
    
    print(f"\n📅 Best Day for Volume: {best_day_posting} ({weekday_stats.loc[best_day_posting, 'postings']:,.0f} postings)")
    print(f"📅 Best Day for Engagement: {best_day_engagement} ({weekday_stats.loc[best_day_engagement, 'apps_per_posting']:.1f} apps/posting)")
    
    # Quarterly trends
    df_filtered['quarter'] = df_filtered['posting_date'].dt.quarter
    quarterly = df_filtered.groupby('quarter').agg({
        'title': 'count',
        'metadata_totalNumberJobApplication': 'sum',
        'average_salary': 'mean'
    }).rename(columns={'title': 'postings'})
    
    print("\n📊 QUARTERLY TRENDS")
    print("-"*80)
    print(quarterly.round(2))
    
    # Recommendations
    print("\n" + "="*80)
    print("💡 POSTING TIMING RECOMMENDATIONS")
    print("="*80)
    
    print(f"\n🎯 Best Time to Post Jobs:")
    print(f"  • Month: {best_posting['month_str']} (highest volume)")
    print(f"  • Day: {best_day_posting} (most postings)")
    print(f"  • Day for Engagement: {best_day_engagement} (most apps per posting)")
    
    print(f"\n⚠️ Avoid posting during:")
    print(f"  • Month: {worst_posting['month_str']} (lowest volume)")
    
    print(f"\n📈 Consider posting on {best_day_posting}s for maximum visibility")
    print(f"   and on {best_day_engagement}s for higher engagement rates")
    
    return monthly, weekday_stats

# Run Q5
q5_posting_timing_numbers(df)

Q5: OPTIMAL POSTING TIMING ANALYSIS

📊 TIME PERIOD OVERVIEW
--------------------------------------------------------------------------------
Data from: 2023-05-01 to 2024-05-29
Total Postings in Period: 981,524

📊 MONTHLY TRENDS
--------------------------------------------------------------------------------
   month_str  posting_count  metadata_totalNumberJobApplication  \
0    2023-05          70213                              706806   
1    2023-06          70410                              439994   
2    2023-07          82878                               19871   
3    2023-08          83766                               17595   
4    2023-09          76481                               16872   
5    2023-10          80131                               55632   
6    2023-11          72402                               15316   
7    2023-12          61039                               14894   
8    2024-01          80851                               24757   
9    2024-02        

(   posting_month  posting_count  metadata_totalNumberJobApplication  \
 0        2023-05          70213                              706806   
 1        2023-06          70410                              439994   
 2        2023-07          82878                               19871   
 3        2023-08          83766                               17595   
 4        2023-09          76481                               16872   
 5        2023-10          80131                               55632   
 6        2023-11          72402                               15316   
 7        2023-12          61039                               14894   
 8        2024-01          80851                               24757   
 9        2024-02          69069                               21297   
 10       2024-03          79511                               26919   
 11       2024-04          81642                               57670   
 12       2024-05          73131                               2

In [7]:
def analyze_seasonal_patterns(df):
    """
    Analyze posting and application patterns by month (aggregated across years)
    To identify when to apply for best outcomes

    POSTING = A Job Advertisement
    It's the job ad you see on a website
    
    One posting = One job listing
    
    Like a "Help Wanted" sign
    
    VACANCY = Number of Positions Available
    How many people the company wants to hire
    
    One posting can have MULTIPLE vacancies
    """
    
    print("="*80)
    print("📅 SEASONAL PATTERNS: WHEN TO APPLY FOR BEST RESULTS")
    print("="*80)
    
    # Convert to datetime
    df['posting_date'] = pd.to_datetime(df['metadata_originalPostingDate'])
    
    # Filter from May 2023
    df_filtered = df[df['posting_date'] >= '2023-05-01'].copy()
    
    # Extract month only (ignoring year)
    df_filtered['month_only'] = df_filtered['posting_date'].dt.month
    
    # Aggregate by month (across all years)
    monthly_agg = df_filtered.groupby('month_only').agg({
        'title': 'count',
        'metadata_totalNumberJobApplication': 'sum',
        'numberOfVacancies': 'sum'
    }).rename(columns={'title': 'postings'})
    
    # Calculate applications per posting
    monthly_agg['apps_per_posting'] = monthly_agg['metadata_totalNumberJobApplication'] / monthly_agg['postings']
    monthly_agg['competition_score'] = monthly_agg['metadata_totalNumberJobApplication'] / monthly_agg['numberOfVacancies']
    
    # Month names
    month_names = {
        1: 'January', 2: 'February', 3: 'March', 4: 'April',
        5: 'May', 6: 'June', 7: 'July', 8: 'August',
        9: 'September', 10: 'October', 11: 'November', 12: 'December'
    }
    monthly_agg['month_name'] = monthly_agg.index.map(month_names)
    
    print("\n📊 MONTHLY PATTERNS (Aggregated)")
    print("-"*80)
    print(monthly_agg[['month_name', 'postings', 'metadata_totalNumberJobApplication', 
                       'apps_per_posting', 'competition_score']].round(2))
    
    # Rankings
    print("\n🏆 MONTHLY RANKINGS")
    print("-"*80)
    
    # Most postings
    most_postings = monthly_agg.nlargest(3, 'postings')
    print(f"\n📌 TOP 3 Months for POSTINGS (Most Jobs Available):")
    for idx, row in most_postings.iterrows():
        print(f"  • {row['month_name']}: {row['postings']:,.0f} postings")
    
    # Most applications
    most_apps = monthly_agg.nlargest(3, 'metadata_totalNumberJobApplication')
    print(f"\n📌 TOP 3 Months for APPLICATIONS (Most Competition):")
    for idx, row in most_apps.iterrows():
        print(f"  • {row['month_name']}: {row['metadata_totalNumberJobApplication']:,.0f} applications")
    
    # Best apps per posting (Engagement)
    best_engagement = monthly_agg.nlargest(3, 'apps_per_posting')
    print(f"\n📌 TOP 3 Months for ENGAGEMENT (Apps per Posting):")
    for idx, row in best_engagement.iterrows():
        print(f"  • {row['month_name']}: {row['apps_per_posting']:.2f} apps/posting")
    
    # Competition intensity
    most_competitive = monthly_agg.nlargest(3, 'competition_score')
    print(f"\n📌 TOP 3 Months for COMPETITION (Apps per Vacancy):")
    for idx, row in most_competitive.iterrows():
        print(f"  • {row['month_name']}: {row['competition_score']:.2f} apps/vacancy")
    
    return monthly_agg

monthly_agg = analyze_seasonal_patterns(df)

📅 SEASONAL PATTERNS: WHEN TO APPLY FOR BEST RESULTS

📊 MONTHLY PATTERNS (Aggregated)
--------------------------------------------------------------------------------
           month_name  postings  metadata_totalNumberJobApplication  \
month_only                                                            
1             January     80851                               24757   
2            February     69069                               21297   
3               March     79511                               26919   
4               April     81642                               57670   
5                 May    143344                              730832   
6                June     70410                              439994   
7                July     82878                               19871   
8              August     83766                               17595   
9           September     76481                               16872   
10            October     80131                      

In [8]:
def analyze_weekly_patterns(df):
    """
    Analyze posting and application patterns by weekday (aggregated)
    """
    
    print("\n" + "="*80)
    print("📆 WEEKLY PATTERNS: WHEN TO APPLY")
    print("="*80)
    
    df['posting_date'] = pd.to_datetime(df['metadata_originalPostingDate'])
    df_filtered = df[df['posting_date'] >= '2023-05-01'].copy()
    
    # Extract weekday (0=Monday, 6=Sunday)
    df_filtered['weekday_num'] = df_filtered['posting_date'].dt.dayofweek
    
    # Aggregate by weekday
    weekday_agg = df_filtered.groupby('weekday_num').agg({
        'title': 'count',
        'metadata_totalNumberJobApplication': 'sum',
        'numberOfVacancies': 'sum'
    }).rename(columns={'title': 'postings'})
    
    weekday_agg['apps_per_posting'] = weekday_agg['metadata_totalNumberJobApplication'] / weekday_agg['postings']
    weekday_agg['competition_score'] = weekday_agg['metadata_totalNumberJobApplication'] / weekday_agg['numberOfVacancies']
    
    # Weekday names
    weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekday_agg['weekday_name'] = weekdays
    
    # Reindex to proper order
    weekday_agg = weekday_agg.reindex(range(7))
    weekday_agg['weekday_name'] = weekdays
    
    print("\n📊 WEEKLY PATTERNS (Aggregated)")
    print("-"*80)
    print(weekday_agg[['weekday_name', 'postings', 'metadata_totalNumberJobApplication', 
                       'apps_per_posting', 'competition_score']].round(2))
    
    # Rankings
    print("\n🏆 WEEKLY RANKINGS")
    print("-"*80)
    
    # Best days to apply
    print(f"\n📌 BEST DAYS TO APPLY:")
    print("  🥇 Tuesday - Most jobs posted, most applications")
    print("  🥈 Wednesday - Best engagement (apps per posting)")
    print("  🥉 Monday/Thursday - Strong secondary options")
    
    # Best engagement
    best_engagement = weekday_agg.nlargest(3, 'apps_per_posting')
    print(f"\n📌 BEST DAYS FOR ENGAGEMENT (Apps per Posting):")
    for idx, row in best_engagement.iterrows():
        print(f"  • {row['weekday_name']}: {row['apps_per_posting']:.2f} apps/posting")
    
    # Competition intensity
    most_competitive = weekday_agg.nlargest(3, 'competition_score')
    print(f"\n📌 MOST COMPETITIVE DAYS (Apps per Vacancy):")
    for idx, row in most_competitive.iterrows():
        print(f"  • {row['weekday_name']}: {row['competition_score']:.2f} apps/vacancy")
    
    return weekday_agg

weekday_agg = analyze_weekly_patterns(df)


📆 WEEKLY PATTERNS: WHEN TO APPLY

📊 WEEKLY PATTERNS (Aggregated)
--------------------------------------------------------------------------------
            weekday_name  postings  metadata_totalNumberJobApplication  \
weekday_num                                                              
0                 Monday    174748                              266216   
1                Tuesday    189737                              302511   
2              Wednesday    170396                              273969   
3               Thursday    179233                              257010   
4                 Friday    181315                              236956   
5               Saturday     48251                               60975   
6                 Sunday     37844                               44012   

             apps_per_posting  competition_score  
weekday_num                                       
0                        1.52               0.56  
1                        1.59    

In [9]:
def complete_seasonal_analysis(df):
    """
    Complete seasonal analysis with actionable recommendations
    """
    
    print("\n" + "="*80)
    print("🌟 COMPLETE SEASONAL ANALYSIS: WHEN TO APPLY")
    print("="*80)
    
    df['posting_date'] = pd.to_datetime(df['metadata_originalPostingDate'])
    df_filtered = df[df['posting_date'] >= '2023-05-01'].copy()
    
    # Extract month
    df_filtered['month_only'] = df_filtered['posting_date'].dt.month
    
    # Monthly patterns
    monthly = df_filtered.groupby('month_only').agg({
        'title': 'count',
        'metadata_totalNumberJobApplication': 'sum',
        'numberOfVacancies': 'sum'
    }).rename(columns={'title': 'postings'})
    
    monthly['apps_per_posting'] = monthly['metadata_totalNumberJobApplication'] / monthly['postings']
    monthly['competition_score'] = monthly['metadata_totalNumberJobApplication'] / monthly['numberOfVacancies']
    monthly['application_rate'] = monthly['metadata_totalNumberJobApplication'] / monthly['postings']
    
    # Month names
    month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
                   7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
    monthly['month_name'] = monthly.index.map(month_names)
    
    # Ranking for each metric
    monthly['rank_postings'] = monthly['postings'].rank(ascending=False)
    monthly['rank_apps'] = monthly['metadata_totalNumberJobApplication'].rank(ascending=False)
    monthly['rank_engagement'] = monthly['apps_per_posting'].rank(ascending=False)
    monthly['rank_competition'] = monthly['competition_score'].rank(ascending=False)
    
    # Create a composite score (average of ranks)
    monthly['composite_score'] = (monthly['rank_postings'] + monthly['rank_apps'] + 
                                   monthly['rank_engagement'] + monthly['rank_competition']) / 4
    
    # Sort by composite score (lower is better)
    monthly_sorted = monthly.sort_values('composite_score')
    
    print("\n📊 MONTHLY COMPOSITE SCORE (Lower = Better)")
    print("-"*80)
    print("Rank based on: Postings + Applications + Engagement + Competition")
    
    for idx, row in monthly_sorted.iterrows():
        print(f"  {row['month_name']}: Composite = {row['composite_score']:.1f}")
        print(f"    • Postings: {row['postings']:,.0f} (#{row['rank_postings']:.0f})")
        print(f"    • Applications: {row['metadata_totalNumberJobApplication']:,.0f} (#{row['rank_apps']:.0f})")
        print(f"    • Engagement: {row['apps_per_posting']:.2f} apps/posting (#{row['rank_engagement']:.0f})")
        print(f"    • Competition: {row['competition_score']:.2f} apps/vacancy (#{row['rank_competition']:.0f})")
    
    # Final recommendations
    print("\n" + "="*80)
    print("🎯 ACTIONABLE RECOMMENDATIONS")
    print("="*80)
    
    # Best months to apply
    best_months = monthly_sorted.head(3)['month_name'].tolist()
    print(f"\n✅ BEST MONTHS TO APPLY (Highest opportunities):")
    for i, month in enumerate(best_months, 1):
        print(f"  {i}. {month}")
    
    print(f"\n   Why? These months have:")
    print(f"   • More job postings")
    print(f"   • More applications (more activity)")
    print(f"   • Higher competition (means more companies hiring)")
    
    # Worst months to apply
    worst_months = monthly_sorted.tail(3)['month_name'].tolist()
    print(f"\n❌ WORST MONTHS TO APPLY (Lowest opportunities):")
    for i, month in enumerate(worst_months, 1):
        print(f"  {i}. {month}")
    
    print(f"\n   Why? These months have:")
    print(f"   • Fewer job postings")
    print(f"   • Lower activity")
    print(f"   • Less competition (fewer companies hiring)")
    
    # Best days to apply
    print(f"\n✅ BEST DAYS TO APPLY:")
    print("  🥇 Tuesday - Most jobs posted (189,737)")
    print("  🥈 Wednesday - Best engagement (1.61 apps/posting)")
    print("  🥉 Monday - Second most applications (266,216)")
    
    print(f"\n❌ WORST DAYS TO APPLY:")
    print("  • Sunday - Lowest activity (37,844 postings)")
    print("  • Saturday - Second lowest (48,251 postings)")
    
    # Monthly summary table
    print("\n📊 QUICK REFERENCE TABLE")
    print("-"*80)
    print("Month   | Postings | Apps | Apps/Posting | Best For")
    print("-"*80)
    
    for month in monthly_sorted.index:
        row = monthly.loc[month]
        best_for = []
        if row['rank_postings'] <= 3:
            best_for.append("Volume")
        if row['rank_apps'] <= 3:
            best_for.append("Activity")
        if row['rank_engagement'] <= 3:
            best_for.append("Engagement")
        if row['rank_competition'] <= 3:
            best_for.append("Competition")
        
        best_text = ", ".join(best_for) if best_for else "Moderate"
        print(f"{row['month_name']:6} | {row['postings']:8,.0f} | {row['metadata_totalNumberJobApplication']:6,.0f} | "
              f"{row['apps_per_posting']:11.2f} | {best_text}")
    
    return monthly_sorted, monthly

complete_seasonal_analysis(df)


🌟 COMPLETE SEASONAL ANALYSIS: WHEN TO APPLY

📊 MONTHLY COMPOSITE SCORE (Lower = Better)
--------------------------------------------------------------------------------
Rank based on: Postings + Applications + Engagement + Competition
  May: Composite = 1.5
    • Postings: 143,344 (#1)
    • Applications: 730,832 (#1)
    • Engagement: 5.10 apps/posting (#2)
    • Competition: 2.00 apps/vacancy (#2)
  Apr: Composite = 3.2
    • Postings: 81,642 (#4)
    • Applications: 57,670 (#3)
    • Engagement: 0.71 apps/posting (#3)
    • Competition: 0.27 apps/vacancy (#3)
  Jun: Composite = 3.5
    • Postings: 70,410 (#10)
    • Applications: 439,994 (#2)
    • Engagement: 6.25 apps/posting (#1)
    • Competition: 2.35 apps/vacancy (#1)
  Oct: Composite = 4.5
    • Postings: 80,131 (#6)
    • Applications: 55,632 (#4)
    • Engagement: 0.69 apps/posting (#4)
    • Competition: 0.25 apps/vacancy (#4)
  Mar: Composite = 5.5
    • Postings: 79,511 (#7)
    • Applications: 26,919 (#5)
    • Engagem

(            postings  metadata_totalNumberJobApplication  numberOfVacancies  \
 month_only                                                                    
 5             143344                              730832             366272   
 4              81642                               57670             212255   
 6              70410                              439994             187225   
 10             80131                               55632             219707   
 3              79511                               26919             206305   
 1              80851                               24757             229991   
 7              82878                               19871             210670   
 2              69069                               21297             187905   
 8              83766                               17595             236293   
 9              76481                               16872             204174   
 12             61039                   

In [16]:
def q6_agency_vs_direct_analysis(df):
    """
    Q6: Agency vs Direct Employer Comparison
    Compares key metrics between agency and direct postings
    """
    
    print("="*80)
    print("Q6: AGENCY VS DIRECT EMPLOYER COMPARISON")
    print("="*80)
    
    # 1. Distribution
    print("\n📊 DISTRIBUTION")
    print("-"*80)
    agency_counts = df['is_agency_post'].value_counts()
    agency_pct = df['is_agency_post'].mean() * 100
    
    print(f"Direct Employer: {agency_counts.get(False, 0):,} ({100-agency_pct:.1f}%)")
    print(f"Agency: {agency_counts.get(True, 0):,} ({agency_pct:.1f}%)")
    
    # 2. Key Metrics Comparison
    print("\n📊 KEY METRICS COMPARISON")
    print("-"*80)
    
    comparison = df.groupby('is_agency_post').agg({
        'title': 'count',
        'average_salary': 'mean',
        'applications_per_vacancy': 'mean',
        'metadata_totalNumberJobApplication': 'sum',
        'is_hard_to_fill': 'mean',
        'numberOfVacancies': 'sum'
    }).round(2)
    
    comparison.columns = ['Postings', 'Avg Salary', 'Avg Apps/Vac', 'Total Apps', 'Hard to Fill %', 'Total Vacancies']
    comparison['Hard to Fill %'] = comparison['Hard to Fill %'] * 100
    comparison.index = ['Direct Employer', 'Agency']
    
    print(comparison)
    
    # 3. Key Differences
    print("\n📊 KEY DIFFERENCES")
    print("-"*80)
    
    if len(df[df['is_agency_post']]) > 0 and len(df[~df['is_agency_post']]) > 0:
        agency = df[df['is_agency_post']]
        direct = df[~df['is_agency_post']]
        
        salary_diff = agency['average_salary'].mean() - direct['average_salary'].mean()
        apps_diff = agency['applications_per_vacancy'].mean() - direct['applications_per_vacancy'].mean()
        htf_diff = agency['is_hard_to_fill'].mean() - direct['is_hard_to_fill'].mean()
        
        print(f"💰 Salary: Agency ${abs(salary_diff):,.0f} {'higher' if salary_diff > 0 else 'lower'} than Direct")
        print(f"📊 Apps/Vacancy: Agency has {abs(apps_diff):.2f} {'more' if apps_diff > 0 else 'fewer'} applications per vacancy")
        print(f"🔴 Hard to Fill: Agency is {abs(htf_diff*100):.1f}% {'more' if htf_diff > 0 else 'less'} likely to be hard to fill")
    
    # 4. By Position Level
    print("\n📊 AGENCY VS DIRECT BY POSITION LEVEL")
    print("-"*80)
    
    position_comparison = df.groupby(['positionLevels', 'is_agency_post']).agg({
        'title': 'count',
        'average_salary': 'mean',
        'applications_per_vacancy': 'mean'
    }).round(2)
    
    position_comparison.columns = ['Postings', 'Avg Salary', 'Avg Apps/Vac']
    position_pivot = position_comparison.unstack(level=1)
    
    print(position_pivot)
    
    # 5. By Industry (Top 10)
    print("\n📊 TOP 10 INDUSTRIES - AGENCY VS DIRECT")
    print("-"*80)
    
    top_industries = df['industry_primary'].value_counts().head(10).index
    industry_comparison = df[df['industry_primary'].isin(top_industries)].groupby(['industry_primary', 'is_agency_post']).agg({
        'title': 'count',
        'average_salary': 'mean'
    }).round(2)
    
    industry_comparison.columns = ['Postings', 'Avg Salary']
    industry_pivot = industry_comparison.unstack(level=1)
    
    print(industry_pivot)
    
    # 6. Monthly Trend (Agency vs Direct)
    print("\n📊 MONTHLY TREND - AGENCY VS DIRECT POSTINGS")
    print("-"*80)
    
    df['posting_month'] = pd.to_datetime(df['metadata_originalPostingDate']).dt.to_period('M')
    monthly_agency = df.groupby(['posting_month', 'is_agency_post']).size().unstack(fill_value=0)
    monthly_agency.columns = ['Direct', 'Agency']
    
    # Show last 6 months
    print(monthly_agency.tail(6))
    
    # 7. Summary Insights
    print("\n" + "="*80)
    print("💡 KEY INSIGHTS")
    print("="*80)
    
    agency_pct = df['is_agency_post'].mean() * 100
    agency_salary = df[df['is_agency_post']]['average_salary'].mean()
    direct_salary = df[~df['is_agency_post']]['average_salary'].mean()
    agency_htf = df[df['is_agency_post']]['is_hard_to_fill'].mean() * 100
    direct_htf = df[~df['is_agency_post']]['is_hard_to_fill'].mean() * 100
    
    print(f"\n📌 Agency posts make up {agency_pct:.1f}% of all job postings")
    print(f"📌 Agency salaries are ${abs(agency_salary - direct_salary):,.0f} {'higher' if agency_salary > direct_salary else 'lower'} than Direct")
    print(f"📌 Agency posts are {abs(agency_htf - direct_htf):.1f}% {'more' if agency_htf > direct_htf else 'less'} likely to be hard to fill")
    
    print("\n📌 Recommendations:")
    if agency_salary > direct_salary:
        print("  • Agency postings offer higher salaries - good for job seekers")
    else:
        print("  • Direct postings offer higher salaries - apply directly")
    
    if agency_htf > direct_htf:
        print("  • Agency postings are harder to fill - may indicate stricter requirements")
    else:
        print("  • Direct postings are harder to fill - may need to reconsider recruitment strategy")
    
    return comparison

# Run Q6
q6_agency_vs_direct_analysis(df)

Q6: AGENCY VS DIRECT EMPLOYER COMPARISON

📊 DISTRIBUTION
--------------------------------------------------------------------------------
Direct Employer: 982,729 (94.1%)
Agency: 61,868 (5.9%)

📊 KEY METRICS COMPARISON
--------------------------------------------------------------------------------
                 Postings  Avg Salary  Avg Apps/Vac  Total Apps  \
Direct Employer    982729     4795.48          1.72     2174475   
Agency              61868     4663.34          0.74       65901   

                 Hard to Fill %  Total Vacancies  
Direct Employer             0.0          2577095  
Agency                      0.0           233158  

📊 KEY DIFFERENCES
--------------------------------------------------------------------------------
💰 Salary: Agency $132 lower than Direct
📊 Apps/Vacancy: Agency has 0.99 fewer applications per vacancy
🔴 Hard to Fill: Agency is 0.0% less likely to be hard to fill

📊 AGENCY VS DIRECT BY POSITION LEVEL
------------------------------------------

,Postings,Avg Salary,Avg Apps/Vac,Total Apps,Hard to Fill %,Total Vacancies
Direct Employer,982729,4795.48,1.72,2174475,0.0,2577095
Agency,61868,4663.34,0.74,65901,0.0,233158
